# SingleBehaviorLab demo — behavior sequencing

This notebook walks through the supervised pipeline: train a classifier on a labeled experiment, run inference on a new video, and inspect the raw predictions.

**Labeling is GUI-only.** Before running this notebook you need a labeled experiment directory with at least a few annotated clips. Create one in the app (*File → New Experiment*), use the Labeling tab to tag clips, and point the `EXPERIMENT` variable below at the resulting folder.

See [`../CLI.md`](../CLI.md) for the equivalent one-shot CLI invocations.

## 1. Configuration

Edit the three paths below to match your setup. The defaults assume you have populated `demo/data/` as described in [`README.md`](README.md).

In [ ]:
from pathlib import Path

EXPERIMENT = Path("data/experiment").resolve()
VIDEO = Path("data/demo.mp4").resolve()
MODEL_NAME = "demo_model"  # saved under EXPERIMENT/models/behavior_heads/demo_model.pt

assert EXPERIMENT.is_dir(), f"Experiment directory not found: {EXPERIMENT}"
assert VIDEO.is_file(), f"Video file not found: {VIDEO}"

print(f"Experiment: {EXPERIMENT}")
print(f"Video:      {VIDEO}")

## 2. Train a behavior classifier

`run_training_session` assembles the model + datasets from the experiment directory and drives one training run. Pass a profile name (e.g. `balanced`) and optional per-run overrides such as `cli_overrides={"epochs": 20}`.

The training loop logs to a Python callable (`log_fn`); here we print each line so the notebook shows progress inline.

In [ ]:
import singlebehaviorlab as sbl

result = sbl.train(
    EXPERIMENT,
    profile="balanced",
    cli_overrides={"epochs": 20},
    output_name=MODEL_NAME,
    log_fn=print,
)

MODEL_PATH = Path(result["output_path"])
print()
print("Checkpoint:      ", MODEL_PATH)
print("Best val acc:    ", result.get("best_val_acc"))
print("Best val F1:     ", result.get("best_val_f1"))

## 3. Run inference on a new video

`run_inference_on_video` reads the trained checkpoint (with its sibling `.meta.json`), processes the video in clip batches, and writes a results JSON in the exact format the **Inference tab → Load results** action consumes. Smoothing, Viterbi decoding, and gap-filling are applied interactively in the GUI, so the CLI/Python API writes the raw model output.

In [ ]:
RESULTS_JSON = VIDEO.with_name(f"{VIDEO.stem}_results.json")

sbl.infer(
    MODEL_PATH,
    VIDEO,
    RESULTS_JSON,
    experiment_dir=EXPERIMENT,
    log_fn=print,
)

print(f"\nResults written to: {RESULTS_JSON}")

## 4. Inspect the raw results

The JSON contains one entry per video under `results[...]`. Each entry has the raw per-clip predictions, per-frame probabilities (when the model has a frame head), and an aggregated per-frame probability matrix built from center-weighted overlap merging.

In [ ]:
import json
import numpy as np

with open(RESULTS_JSON) as f:
    payload = json.load(f)

classes = payload["classes"]
video_key = next(iter(payload["results"]))
entry = payload["results"][video_key]

print(f"Classes:       {classes}")
print(f"Total frames:  {entry['total_frames']}")
print(f"Clip count:    {len(entry['predictions'])}")
print(f"Original FPS:  {entry['orig_fps']:.2f}")

agg = entry.get("aggregated_frame_probs")
if agg is not None:
    agg = np.asarray(agg)
    print(f"Per-frame prob matrix shape: {agg.shape}")
    per_class_means = agg.mean(axis=0)
    for name, score in zip(classes, per_class_means):
        print(f"  mean prob — {name:>24s}: {score:.3f}")

## 5. Plot the ethogram

A quick matplotlib rendering of the aggregated per-frame class probabilities. For smoothing and segment merging, load the JSON in the GUI's Inference tab instead.

In [ ]:
import matplotlib.pyplot as plt

if agg is None:
    print("Aggregated frame probabilities not available for this model.")
else:
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.imshow(
        agg.T,
        aspect="auto",
        interpolation="nearest",
        extent=[0, agg.shape[0], 0, len(classes)],
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_yticks([i + 0.5 for i in range(len(classes))])
    ax.set_yticklabels(classes)
    ax.set_xlabel("Frame")
    ax.set_title(f"Ethogram — {video_key.split('/')[-1]}")
    fig.tight_layout()
    plt.show()

## Next steps

- Open `RESULTS_JSON` in the GUI (**Inference tab → Load results**) to apply temporal smoothing, Viterbi decoding, gap filling, and to review flagged uncertain clips.
- Use the **Refinement** tab to accept or reassign uncertain predictions, then retrain with more data.
- See [`02_segmentation_clustering.ipynb`](02_segmentation_clustering.ipynb) for the unsupervised discovery pipeline.